In [0]:
%sql
-- Verifying the number of records in each table
SELECT 'Products' AS TableName, COUNT(*) AS Records FROM Products UNION ALL SELECT 'Customers', COUNT(*) FROM Customers UNION ALL SELECT 'Sales', COUNT(*) FROM Sales UNION ALL SELECT 'Customer_Feedback', COUNT(*) FROM Customer_Feedback;

--Q1.1
SELECT 
  productID, 
  productname, 
  UnitPrice
FROM products
ORDER BY UnitPrice DESC

--Q1.2
SELECT region, 
COUNT (CustomerID) as Count 
FROM customers 
GROUP BY region ORDER BY count DESC

--Q1.3 
SELECT OrderID, 
OrderDate, 
TotalSales
FROM sales
Order by OrderDate DESC
LIMIT 10

--Q1.4
SELECT ProductName, 
Category, 
UnitPrice
FROM Products
WHERE UnitPrice < 1000

--Q1.5
SELECT Satisfaction, 
COUNT(CustomerID) AS SatisfatcionCount
FROM Customer_Feedback
GROUP BY Satisfaction
ORDER BY SatisfatcionCount DESC;

--Q2.1
SELECT p.Category, 
SUM(s.TotalSales) as TotalRevenue, 
SUM(S.pROFIT) as TotalProfit
FROM workspace.default.sales s
INNER JOIN workspace.default.products p
ON s.ProductID = p.ProductID
GROUP BY p.Category
ORDER BY TotalRevenue DESC

--Q2.2
SELECT C.CustomerID, 
CONCAT(C.FirstName, ' ', C.LastName) AS CustomerName, 
ROUND(SUM(S.TotalSales), 2) AS TotalSpent
FROM customers c
JOIN sales S
ON C.CustomerID = S.CustomerID
GROUP BY C.CustomerID, CustomerName
ORDER BY TotalSpent DESC
LIMIT 5

--Q2.3
SELECT EXTRACT(YEAR FROM OrderDate) AS Year,
EXTRACT(MONTH FROM OrderDate) AS Month,
ROUND(SUM(TotalSales), 2) AS TotalSales
FROM Sales
WHERE EXTRACT(YEAR FROM OrderDate) = 2024
GROUP BY 
EXTRACT(YEAR FROM OrderDate),
EXTRACT(MONTH FROM OrderDate)
ORDER BY 
Month ASC

--Q2.4
SELECT Channel,
COUNT(OrderID) AS NumberOfOrders,
ROUND(AVG(TotalSales), 2) AS AveragOrderValue,
ROUND(SUM(TotalSales), 2) AS TotalRevenue 
FROM Sales
GROUP BY Channel
ORDER BY TotalRevenue DESC;

--Q2.5
SELECT p.Category, 
ROUND(AVG(cf.Rating), 2) AS AverageRatings,
COUNT(cf.FeedbackID) AS NumberOfReviews
FROM Customer_Feedback cf
JOIN Sales s
ON cf.OrderID = s.OrderID
JOIN Products p
ON s.ProductID = p.ProductID
GROUP BY p.Category
HAVING COUNT(cf.FeedbackID) >= 50
ORDER BY AverageRatings DESC;

--Q3.1
WITH ProductSales AS (
    SELECT 
    p.Category, 
    p.ProductName, 
    SUM(s.UnitPrice) AS TotalRevenue
    FROM Products p
    JOIN Sales s ON p.ProductID = s.ProductID
    GROUP BY p.Category, p.ProductName
),
RankedProducts AS (
    SELECT 
    category,
    ProductName, 
    TotalRevenue, 
    ROW_NUMBER() OVER (PARTITION BY Category 
    ORDER BY TotalRevenue DESC
    ) AS rn
    FROM ProductSales
)
SELECT 
Category, 
ProductName, 
TotalRevenue
FROM RankedProducts
WHERE rn = 1
ORDER BY Category;

--Q3.2
SELECT c.CustomerID, 
c.FirstName AS CustomerName,
c.Region,c.Channel AS PrimaryChannel,
SUM(UnitPrice) AS TotalPurchases,
COUNT(s.OrderID) AS NumberOfOrders,
ROUND(AVG(s.UnitPrice), 2) AS AverageOrderValue
FROM Customers c
JOIN Sales s
ON c.CustomerID = s.CustomerID
GROUP BY c.CustomerID, 
c.FirstName, 
c.LastName, 
c.Region, 
c.Channel
HAVING COUNT(s.OrderID) > 3
ORDER BY TotalPurchases DESC

--Q3.3
SELECT 
    p.ProductName,
    p.Category,
    SUM(s.UnitPrice) AS TotalSales,
    SUM(s.Profit) AS TotalProfit,
    ROUND((SUM(s.Profit) / SUM(s.UnitPrice)) * 100, 2) AS ProfitMargin
FROM Products p
JOIN Sales s
ON p.ProductID = s.ProductID
GROUP BY p.ProductName, p.Category
ORDER BY ProfitMargin DESC;

--Q3.4
WITH YearlySales AS (
SELECT YEAR(OrderDate) AS SalesYear, 
SUM(UnitPrice) AS TotalSales
FROM Sales
WHERE YEAR(OrderDate) IN (2023, 2024)
GROUP BY YEAR(OrderDate)
)
SELECT 
y2023.TotalSales AS Year2023,
y2024.TotalSales AS Year2024,
((y2024.TotalSales - y2023.TotalSales) / y2023.TotalSales) * 100 AS GrowthPercentage
FROM 
(SELECT TotalSales FROM YearlySales WHERE SalesYear = 2023) y2023,
(SELECT TotalSales FROM YearlySales WHERE SalesYear = 2024) y2024

--Q3.5
WITH RegionSales AS (
    SELECT c.Region, 
    SUM(s.UnitPrice) AS TotalSales, 
    COUNT(s.OrderID) AS TotalOrders
    FROM Customers c
    JOIN Sales s
    ON c.CustomerID = s.CustomerID
    GROUP BY c.Region
)
SELECT Region, TotalSales,TotalOrders,
ROW_NUMBER() OVER (ORDER BY TotalSales DESC) AS Rank
FROM RegionSales
ORDER BY Rank

--Q4.1
SELECT
  CASE 
    WHEN rating BETWEEN 4 AND 5 THEN 'Highly Satisfied'
    WHEN rating BETWEEN 1 AND 3 THEN 'Less Satisfied'
    ELSE 'neutral'
  END AS satisfaction_level,
  AVG(order_count) AS avg_orders_per_customer,
  COUNT(DISTINCT customerID) AS total_customers
FROM (
  SELECT
    s.CustomerID,
    MAX(cf.rating) AS rating,
    COUNT(s.OrderID) AS order_count
  FROM workspace.default.sales s
  INNER JOIN workspace.default.customer_feedback cf
    ON s.CustomerID = cf.CustomerID
  GROUP BY s.CustomerID
  HAVING MAX(cf.rating) BETWEEN 1 AND 5
)
GROUP BY satisfaction_level

--Q4.2
SELECT 
  CASE 
    WHEN DiscountPercent = 0 THEN '0%'
    WHEN DiscountPercent BETWEEN 1 AND 10 THEN '1-10%'
    WHEN DiscountPercent BETWEEN 11 AND 20 THEN '11-20%'
    WHEN DiscountPercent BETWEEN 21 AND 30 THEN '21-30%'
    ELSE '31%+'
  END AS discount_band,
  COUNT(*) AS order_count,
  ROUND(SUM(TotalSales), 2) AS total_sales,
  ROUND(SUM(Profit), 2) AS total_profit,
  ROUND(SUM(Profit) / SUM(TotalSales) * 100, 2) AS profit_margin_pct
FROM sales
GROUP BY discount_band
ORDER BY 
  CASE discount_band
    WHEN '0%' THEN 1
    WHEN '1-10%' THEN 2
    WHEN '11-20%' THEN 3
    WHEN '21-30%' THEN 4
    ELSE 5
  END


--Q4.3
SELECT
  p.ProductID,
  p.ProductName,
  ROUND(SUM(s.TotalSales), 2) AS total_revenue,
  SUM(s.Quantity) AS sales_volume
FROM
  products p
JOIN
  sales s ON p.ProductID = s.ProductID
GROUP BY
  p.ProductID, p.ProductName
ORDER BY
  total_revenue ASC
LIMIT 5



TableName,Records
Products,30
Customers,500
Sales,2500
Customer_Feedback,1000
